In [6]:
import pandas as pd
import numpy as np
from thefuzz import process, fuzz

# ==========================================
# 0. MEMBACA DATASET ASLI (CSV)
# ==========================================
print("📂 Membaca dataset dari file CSV...")

# Ganti nama file ini jika berbeda dengan yang ada di komputer Anda
df_rawg = pd.read_excel("Dataset_Game_RAWG.xlsx")
df_rad_ps = pd.read_excel("Dataset_RAD_Playstation.xlsx")

print(f"Total data RAWG (Global): {len(df_rawg)} baris")
print(f"Total data RAD PS (Lokal): {len(df_rad_ps)} baris")

# ==========================================
# 1. FUZZY STRING MATCHING
# ==========================================
print("\n⚙️ Memulai proses pencocokan nama game (Fuzzy Matching)...")

matched_titles = {}
# Mengambil daftar semua judul global dan memastikan tipenya string (menghindari error float/NaN)
daftar_judul_global = df_rawg['Judul'].astype(str).tolist()

# Batas toleransi kemiripan (0-100). Angka 70 biasanya ideal.
THRESHOLD_KEMIRIPAN = 70 

# Memastikan kolom Judul_Game berupa string
for judul_rental in df_rad_ps['Judul_Game'].astype(str):
    # Mencari kecocokan terbaik menggunakan fuzz.token_set_ratio
    best_match, score = process.extractOne(judul_rental, daftar_judul_global, scorer=fuzz.token_set_ratio)
    
    if score >= THRESHOLD_KEMIRIPAN:
        matched_titles[judul_rental] = best_match
        # Bisa di-uncomment baris di bawah ini jika ingin melihat proses match satu per satu di terminal
        # print(f"✔️ MATCH: '{judul_rental}' --> '{best_match}' (Skor: {score})")

# Buat kolom baru di df_rad_ps untuk menyimpan nama resmi RAWG hasil pencocokan
df_rad_ps['Judul_RAWG_Matched'] = df_rad_ps['Judul_Game'].map(matched_titles)

print(f"Selesai! {len(matched_titles)} game dari rental berhasil dicocokkan dengan data global.")

# ==========================================
# 2. STATUS INVENTARIS & GABUNGKAN DATA (MERGE)
# ==========================================
print("\n🔗 Menggabungkan dataset...")
# Melakukan Left Join: Semua data RAWG tetap ada, data RAD menempel jika cocok
df_final = pd.merge(
    df_rawg, 
    df_rad_ps[['Judul_RAWG_Matched', 'Total_Sewa', 'Size_Aktual_GB']], 
    left_on='Judul', 
    right_on='Judul_RAWG_Matched', 
    how='left'
)

# Membuat kolom Status Inventaris: Jika 'Judul_RAWG_Matched' tidak kosong, berarti 'Ada'
df_final['Status_Inventaris'] = np.where(df_final['Judul_RAWG_Matched'].notna(), 'Ada', 'Tidak Ada')

# ==========================================
# 3. DATA RECONCILIATION (PRIORITAS GROUND TRUTH)
# ==========================================
print("⚖️ Melakukan rekonsiliasi data (Memperbarui Size GB)...")
# Logika: Jika Size_Aktual_GB (dari rental) tidak kosong, ganti/override nilai Size_GB. 
# Jika kosong, biarkan menggunakan Size_GB lama (Sintetik).
df_final['Size_GB'] = np.where(
    df_final['Size_Aktual_GB'].notna(), 
    df_final['Size_Aktual_GB'], # Pakai data Aktual Rental
    df_final['Size_GB']         # Pakai data Sintetik RAWG
)

# Membersihkan kolom-kolom sementara / referensi agar dataset kembali rapi
df_final = df_final.drop(columns=['Judul_RAWG_Matched', 'Size_Aktual_GB'])

# Mengisi NaN pada Total_Sewa dengan 0 untuk game yang tidak ada di rental
df_final['Total_Sewa'] = df_final['Total_Sewa'].fillna(0)

# ==========================================
# 4. SIMPAN HASIL AKHIR
# ==========================================
nama_file_output = "Dataset_Final_DSS_Rental.csv"
df_final.to_csv(nama_file_output, index=False)

print(f"\n✅ BERHASIL! Dataset final telah disimpan sebagai '{nama_file_output}'")
display(df_final[['Judul', 'Status_Inventaris', 'Size_GB', 'Total_Sewa']].head(10))

📂 Membaca dataset dari file CSV...
Total data RAWG (Global): 2000 baris
Total data RAD PS (Lokal): 28 baris

⚙️ Memulai proses pencocokan nama game (Fuzzy Matching)...
Selesai! 24 game dari rental berhasil dicocokkan dengan data global.

🔗 Menggabungkan dataset...
⚖️ Melakukan rekonsiliasi data (Memperbarui Size GB)...

✅ BERHASIL! Dataset final telah disimpan sebagai 'Dataset_Final_DSS_Rental.csv'


,Judul,Status_Inventaris,Size_GB,Total_Sewa
0,EA SPORTS FC 26,Ada,45.00,694.0
1,EA SPORTS FC 25,Tidak Ada,39.70,0.0
2,E-Football,Ada,50.00,454.0
3,NBA 2K26,Ada,110.00,190.0
4,NBA 2K26,Ada,105.00,191.0
5,Tekken 8,Ada,80.00,514.0
6,Street Fighter 6,Tidak Ada,102.39,0.0
7,Mortal Kombat 1,Ada,100.00,115.0
8,Gran Turismo 7,Tidak Ada,49.89,0.0
9,Crash Team Racing Nitro-Fueled,Tidak Ada,79.38,0.0


In [10]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from thefuzz import process, fuzz # Pastikan thefuzz sudah terinstal

# ==========================================
# 1. LOAD DATASET FINAL
# ==========================================
print("📂 Membaca dataset final...")
# Ganti dengan nama file dataset final Anda
df = pd.read_csv("Dataset_Final_DSS_Rental.csv") 

# Pengecekan nama kolom baru
if 'Bisa_PS4' not in df.columns or 'Bisa_PS5' not in df.columns:
    print("⚠️ Peringatan: Kolom 'Bisa_PS4' atau 'Bisa_PS5' tidak ditemukan di dataset.")

# ==========================================
# 2. PREPROCESSING UNTUK MODELING
# ==========================================
print("⚙️ Membangun Matriks Fitur...")
df_model = df.copy()

# --- TAMBAHKAN BAGIAN INI ---
# Mengubah teks 'Yes'/'No' menjadi angka biner 1 dan 0 agar bisa dihitung
if 'Local_Multiplayer' in df_model.columns:
    df_model['Local_Multiplayer'] = df_model['Local_Multiplayer'].map({'Yes': 1, 'No': 0})
# ----------------------------

# One-Hot Encoding untuk Genre (Mengubah teks 'Action, RPG' menjadi angka biner 0 dan 1)
if 'Genre' in df_model.columns:
    genres_split = df_model['Genre'].astype(str).str.get_dummies(sep=', ')
    df_model = pd.concat([df_model, genres_split], axis=1)

# Pilih fitur-fitur numerik yang akan dijadikan acuan kemiripan (Similarity)
fitur_numerik = ['Rating_Global', 'Waktu_Main_Jam', 'Size_GB', 'Local_Multiplayer']
fitur_lengkap = fitur_numerik + list(genres_split.columns)

# Mengisi data kosong (NaN) dengan 0 agar kalkulator tidak error
matriks_mentah = df_model[fitur_lengkap].fillna(0)

# ==========================================
# 3. NORMALISASI & COSINE SIMILARITY
# ==========================================
print("⚖️ Melakukan Normalisasi Skala Min-Max...")
scaler = MinMaxScaler()
matriks_normalisasi = scaler.fit_transform(matriks_mentah)

print("🧠 Menghitung Item-Based Collaborative Filtering (Cosine Similarity)...")
similarity_matrix = cosine_similarity(matriks_normalisasi)

sim_df = pd.DataFrame(similarity_matrix, index=df['Judul'], columns=df['Judul'])
print("✅ Model Cosine Similarity siap digunakan!\n")

# ==========================================
# 4. FUNGSI DECISION SUPPORT SYSTEM (DSS)
# ==========================================
def rekomendasi_tambah_aset(judul_game_input, top_n=5):
    # A. FUZZY SEARCH
    daftar_judul = df['Judul'].astype(str).tolist()
    best_match, score = process.extractOne(judul_game_input, daftar_judul, scorer=fuzz.token_set_ratio)
    
    if score < 70:
        return f"❌ Game '{judul_game_input}' tidak dikenali di dalam database."
    
    judul_resmi = best_match
    print(f"🔎 Game Target: '{judul_resmi}' (Skor Pencarian: {score})")
    print("-" * 60)
    
    # B. AMBIL SKOR KEMIRIPAN
    skor_kemiripan = sim_df[judul_resmi].reset_index()
    skor_kemiripan.columns = ['Judul', 'Skor_Kemiripan']
    
    # C. GABUNGKAN DENGAN METADATA
    df_kandidat = pd.merge(
        skor_kemiripan, 
        df[['Judul', 'Status_Inventaris', 'Bisa_PS4', 'Bisa_PS5', 'Genre']], 
        on='Judul', 
        how='left'
    )
    
    # Membuang game itu sendiri dari rekomendasi
    df_kandidat = df_kandidat[df_kandidat['Judul'] != judul_resmi]
    
    print(f"[DEBUG] Jumlah awal game mirip: {len(df_kandidat)}")
    
    # ----------------------------------------------------
    # D. PENERAPAN FILTER BISNIS DENGAN RADAR DEBUG
    # ----------------------------------------------------
    
    # Filter 1: HANYA game yang BELUM ADA
    # Kita tambahkan .astype(str).str.strip() untuk membersihkan spasi tersembunyi
    kondisi_belum_ada = df_kandidat['Status_Inventaris'].astype(str).str.strip().str.lower() != 'ada'
    df_kandidat = df_kandidat[kondisi_belum_ada]
    print(f"[DEBUG] Sisa setelah Filter 1 (Dibuang karena sudah di inventaris): {len(df_kandidat)}")
    
    # Filter 2: HANYA PS4/PS5 (Sekarang kebal terhadap angka/teks/boolean)
    valid_values = ['1', '1.0', 'yes', 'true', 'ya']
    kondisi_ps4 = df_kandidat['Bisa_PS4'].astype(str).str.strip().str.lower().isin(valid_values)
    kondisi_ps5 = df_kandidat['Bisa_PS5'].astype(str).str.strip().str.lower().isin(valid_values)
    
    df_kandidat = df_kandidat[kondisi_ps4 | kondisi_ps5]
    print(f"[DEBUG] Sisa setelah Filter 2 (Dibuang karena bukan PS4/PS5): {len(df_kandidat)}")
    
    # ----------------------------------------------------
    # E. PEMFORMATAN & PENGURUTAN OUTPUT
    # ----------------------------------------------------
    hasil_rekomendasi = df_kandidat.sort_values(by='Skor_Kemiripan', ascending=False).head(top_n)
    hasil_rekomendasi = hasil_rekomendasi.reset_index(drop=True)
    hasil_rekomendasi.index += 1 
    
    if hasil_rekomendasi.empty:
        return "\n⚠️ Tidak ada rekomendasi yang memenuhi kriteria.\nBisa jadi semua game yang mirip sudah dimiliki oleh rental, atau bukan game PS4/PS5."
    
    hasil_rekomendasi['Skor_Kemiripan'] = (hasil_rekomendasi['Skor_Kemiripan'] * 100).round(2).astype(str) + "%"
    
    def label_platform(row):
        platforms = []
        is_ps4 = str(row['Bisa_PS4']).strip().lower() in valid_values
        is_ps5 = str(row['Bisa_PS5']).strip().lower() in valid_values
        if is_ps4: platforms.append('PS4')
        if is_ps5: platforms.append('PS5')
        return ', '.join(platforms)
        
    hasil_rekomendasi['Platform_Tersedia'] = hasil_rekomendasi.apply(label_platform, axis=1)
        
    return hasil_rekomendasi[['Judul', 'Skor_Kemiripan', 'Genre', 'Platform_Tersedia']]

# Jalankan ulang!
hasil = rekomendasi_tambah_aset("FC 26")
display(hasil)

📂 Membaca dataset final...
⚙️ Membangun Matriks Fitur...
⚖️ Melakukan Normalisasi Skala Min-Max...
🧠 Menghitung Item-Based Collaborative Filtering (Cosine Similarity)...
✅ Model Cosine Similarity siap digunakan!

🔎 Game Target: 'EA SPORTS FC 26' (Skor Pencarian: 100)
------------------------------------------------------------
[DEBUG] Jumlah awal game mirip: 2002
[DEBUG] Sisa setelah Filter 1 (Dibuang karena sudah di inventaris): 1977
[DEBUG] Sisa setelah Filter 2 (Dibuang karena bukan PS4/PS5): 1977


,Judul,Skor_Kemiripan,Genre,Platform_Tersedia
1,NBA 2K21,99.97%,"Simulation, Sports","PS4, PS5"
2,EA SPORTS FC 25,99.83%,"Simulation, Sports",PS5
3,PGA TOUR 2K21,99.78%,"Simulation, Sports",PS4
4,PRO EVOLUTION SOCCER 2019,99.77%,"Simulation, Sports",PS4
5,Assetto Corsa Competizione,88.66%,"Racing, Simulation, Sports",PS4


In [11]:
import pandas as pd
import numpy as np
import random

# ==========================================
# 0. SIMULASI DATA (Anggap ini sudah ada di environment Anda)
# ==========================================
# Asumsi: sim_df adalah Item-Item Similarity Matrix (DataFrame) yang sudah Anda buat di tahap sebelumnya
games = ['EA SPORTS FC 26', 'Tekken 8', 'Grand Theft Auto V', 'The Witcher 3', 'eFootball 2024', 'NBA 2K26']
# Membuat matriks similarity dummy untuk keperluan testing script
sim_df = pd.DataFrame(np.random.rand(6, 6), index=games, columns=games)
np.fill_diagonal(sim_df.values, 1.0) # Kemiripan dengan diri sendiri adalah 1

# Dataset Transaksi Rental RAD PS (Data Testing sesungguhnya)
# Berisi ID Pelanggan dan game apa saja yang pernah mereka sewa
data_transaksi = {
    'User_ID': ['U01', 'U01', 'U01', 'U02', 'U02', 'U02', 'U03', 'U03', 'U03'],
    'Judul_Game': ['EA SPORTS FC 26', 'Tekken 8', 'eFootball 2024', 
                   'The Witcher 3', 'Grand Theft Auto V', 'Tekken 8',
                   'NBA 2K26', 'EA SPORTS FC 26', 'eFootball 2024']
}
df_transaksi = pd.DataFrame(data_transaksi)

# ==========================================
# 1. FUNGSI REKOMENDASI BERDASARKAN PROFIL USER
# ==========================================
def get_user_recommendations(user_history, sim_matrix, top_k=3):
    """
    Menghasilkan rekomendasi berdasarkan daftar game yang pernah dimainkan user.
    """
    # Ambil skor similarity untuk semua game yang pernah dimainkan user
    # Lalu hitung rata-rata skor kemiripannya terhadap semua game lain
    skor_kandidat = sim_matrix[user_history].mean(axis=1)
    
    # Hapus game yang sudah dimainkan dari daftar kandidat rekomendasi
    skor_kandidat = skor_kandidat.drop(labels=user_history, errors='ignore')
    
    # Ambil Top-K game dengan skor rata-rata tertinggi
    rekomendasi = skor_kandidat.sort_values(ascending=False).head(top_k).index.tolist()
    return rekomendasi

# ==========================================
# 2. EVALUASI LEAVE-ONE-OUT (MENGHITUNG METRIK)
# ==========================================
print("🔄 Memulai Evaluasi Model (Leave-One-Out Cross-Validation)...\n")

K = 3 # Kita atur Top-K recommendations (K = 3)
hits = 0
total_users_evaluated = 0
precision_scores = []
recall_scores = []

# Mengelompokkan riwayat sewa berdasarkan User_ID
user_groups = df_transaksi.groupby('User_ID')['Judul_Game'].apply(list)

for user_id, history in user_groups.items():
    # Syarat: User minimal harus punya 2 riwayat sewa agar bisa disembunyikan 1
    if len(history) < 2:
        continue
        
    total_users_evaluated += 1
    
    # 1. Sembunyikan 1 game secara acak (Target / Ground Truth)
    target_game = random.choice(history)
    
    # 2. Sisanya menjadi data latih user (User Profile)
    user_profile = [game for game in history if game != target_game]
    
    # 3. Minta rekomendasi dari sistem berdasarkan profil user
    rekomendasi_top_k = get_user_recommendations(user_profile, sim_df, top_k=K)
    
    # 4. Cek apakah game yang disembunyikan muncul di Top-K
    is_hit = 1 if target_game in rekomendasi_top_k else 0
    hits += is_hit
    
    # 5. Hitung Precision dan Recall untuk user ini
    # Karena kita hanya menyembunyikan 1 item relevan (Leave-One-Out):
    # Precision@K = Jumlah tebakan benar / K
    precision_u = is_hit / K
    
    # Recall@K = Jumlah tebakan benar / Total item relevan yang disembunyikan (yaitu 1)
    recall_u = is_hit / 1.0 
    
    precision_scores.append(precision_u)
    recall_scores.append(recall_u)

# ==========================================
# 3. HASIL AKHIR
# ==========================================
hit_ratio = hits / total_users_evaluated
avg_precision = np.mean(precision_scores)
avg_recall = np.mean(recall_scores)

print("-" * 40)
print(f"📊 HASIL EVALUASI METRIK (Top-{K} Recommendations)")
print("-" * 40)
print(f"Total User Dievaluasi : {total_users_evaluated} users")
print(f"Hit Ratio             : {hit_ratio * 100:.2f}%")
print(f"Precision@{K}           : {avg_precision * 100:.2f}%")
print(f"Recall@{K}              : {avg_recall * 100:.2f}%")

🔄 Memulai Evaluasi Model (Leave-One-Out Cross-Validation)...

----------------------------------------
📊 HASIL EVALUASI METRIK (Top-3 Recommendations)
----------------------------------------
Total User Dievaluasi : 3 users
Hit Ratio             : 100.00%
Precision@3           : 33.33%
Recall@3              : 100.00%


In [12]:
import pandas as pd

# 1. Baca data yang sudah kita bersihkan
df = pd.read_csv("Dataset_Final_DSS_Rental.csv")

# 2. Ambil daftar game yang SUDAH DIMILIKI oleh rental (Status = Ada)
df_rental = df[df['Status_Inventaris'] == 'Ada'].copy()

# 3. Buat Logika Keputusan (Decision Logic) berbasis Data Aktual Rental
def tentukan_status_aset(row):
    # Kriteria 1: Game sangat laku di rental ATAU Rating sangat tinggi
    if row['Total_Sewa'] > df_rental['Total_Sewa'].median() or row['Rating_Global'] >= 4.5:
        return "PERTAHANKAN (Aset Berharga)"
    
    # Kriteria 2: Game memakan memori sangat besar TAPI jarang disewa dan ratingnya jelek
    elif row['Size_GB'] > 80 and row['Total_Sewa'] < df_rental['Total_Sewa'].quantile(0.25):
        return "HAPUS (Beban Storage)"
        
    # Sisanya
    else:
        return "MONITOR (Evaluasi bulan depan)"

# Terapkan logika ke dalam data
df_rental['Saran_Sistem_DSS'] = df_rental.apply(tentukan_status_aset, axis=1)

# 4. Susun kolom agar rapi untuk dibaca oleh orang awam (Pemilik Rental)
kolom_laporan = ['Judul', 'Genre', 'Size_GB', 'Rating_Global', 'Total_Sewa', 'Saran_Sistem_DSS']
laporan_eksekutif = df_rental[kolom_laporan].sort_values(by='Total_Sewa', ascending=False)

# 5. Simpan ke Excel
nama_file_laporan = "Laporan_UAT_RAD_Playstation.xlsx"
laporan_eksekutif.to_excel(nama_file_laporan, index=False)

print(f"✅ Laporan berhasil dibuat: {nama_file_laporan}")
print("\nCuplikan Laporan (Top 5 Game Berdasarkan Total Sewa):")
display(laporan_eksekutif.head())

✅ Laporan berhasil dibuat: Laporan_UAT_RAD_Playstation.xlsx

Cuplikan Laporan (Top 5 Game Berdasarkan Total Sewa):


,Judul,Genre,Size_GB,Rating_Global,Total_Sewa,Saran_Sistem_DSS
0,EA SPORTS FC 26,"Simulation, Sports",45.0,3.08,694.0,PERTAHANKAN (Aset Berharga)
76,Mortal Kombat X,"Action, Fighting",45.0,3.86,559.0,PERTAHANKAN (Aset Berharga)
5,Tekken 8,"Action, Fighting",80.0,4.03,514.0,PERTAHANKAN (Aset Berharga)
198,Ghost of Tsushima,"Adventure, Action, RPG",50.0,4.41,470.0,PERTAHANKAN (Aset Berharga)
201,Overcooked,"Casual, Indie, Arcade, Simulation",10.0,4.07,470.0,PERTAHANKAN (Aset Berharga)
